# 1. Bronze Layer
in this layer there is no transformation, just ingestion data from source (csv file)


In [0]:
%sql
-- due to the use of medallion architecture (bronze, silver, and gold layer), it is neccessary to check the existing sceheme. If the there is no scheme, we will create one.
--
show databases

In [0]:
# Creating bronze layer
# after checking scheme, it turned out that the scheme already existed, so we could immediatly proceed with creating table start from bronze layer (raw data)

df = spark.read.csv('/Workspace/Users/ahmadkurniawan2032@gmail.com/ETL-Pipeline-and-Analysis-Car-Rent-Using-Databricks/vehicle_rental_2024_100cars.csv', header=True, inferSchema=True)
df.write.mode('overwrite').saveAsTable('bronze.rental_cars_vehicle')

# 2. Silver Layer

in this layer there is transformation, before transformation we need check quality of the data to understand what kind of transformation that will do with this data

## 2.1 Quality Data Checking

### 2.1.1 Cek Samples of Raw Data

In [0]:
%sql

select * from bronze.rental_cars_vehicle
limit(5)

### 2.1.2 Check Statistic of Data

In [0]:
df.describe().show(vertical=True)

### 2.1.3 Check Suitable of Data Type

In [0]:
df.printSchema()

### 2.1.4 Missing Value Check

In [0]:
from pyspark.sql.functions import col, sum, isnan, when

missing_values = df.select([
    sum(when(col(c).isNull(), 1).otherwise(0)).alias(c) for c in df.columns
]).show(10, vertical=True)

missing_values

### 2.1.5 Duplicate Check

In [0]:
from pyspark.sql.functions import count

duplicates = df.groupBy('transaction_id').agg(count('*').alias('count')).filter(col('count') > 1).count()

print(f"Number of duplicate rows: {duplicates}")

### 2.1.6 Cek Unique Value

In [0]:
from pyspark.sql.functions import countDistinct

unique_values = df.select([
    countDistinct(c).alias(c) for c in df.columns
]).show(10, vertical=True)

unique_values

### 2.1.7 Cek Standarization  

In [0]:
# Cek Standarization of Name Spelling Must not be Lowercase 

from pyspark.sql.functions import col, lower, upper

name_lower = df.filter(col('customer_name')== lower(col('customer_name')))
name_lower.lower(vertical=True)


In [0]:
# cek standarization of name spelling must not be upper case
name_upper = df.filter(col('customer_name')== upper(col('customer_name')))
name_upper.show(vertical=True)

In [0]:
# cek speeling of city name

df.select('customer_city').distinct().show()

In [0]:
# check pick-up location 
df.select('pickup_location').distinct().show(26)


In [0]:
# check return location

df.select('return_location').distinct().show(26)


In [0]:
# check gender speeling

df.select('customer_gender').distinct().show()

In [0]:
# cek email validation
from pyspark.sql.functions import col, instr

df.select('customer_email').filter(
    (instr(col('customer_email'), '@') == 0) |
    (instr(col('customer_email'), '.') == 0) 
).show()




In [0]:
# cek phone number validation
from pyspark.sql.functions import col, regexp_replace

df.withColumn('clean_phone', regexp_replace(col('customer_phone'), '[0-9]', ''))\
    .filter(col('clean_phone') != '')\
    .select('customer_phone', 'clean_phone').show()

### 2.1.8 Suitbale sum of include table and driver name

In [0]:
%sql
--  check date include_driver and name
-- sum of include_driver = false and driver_name = none must equal
select
  count(case when include_driver = false then 1 end) as include_driver,
  count(case when driver_name ilike 'none' then 1 end) as driver_name
from 
  bronze.rental_cars_vehicle

### 2.1.9 Cek Suitable Sum of is_late with penalty_fee  

In [0]:
%sql
-- check sum of is_late = true must equal with sum of penalty_fee =! None
select
  count(case when is_late_return = True then 1 end) as is_late,
  count(case when penalty_fee != 0 then 1 end) as penalty_fee
from
  bronze.rental_cars_vehicle


### 2.1.10 Cek total_amount validation

In [0]:
%sql
-- # check total validation
-- ## total validation must equal with daily_rate * rental_duration_days * (1 - discount_percent / 100) + penalty_fee

select  
  is_late_return,
  total_amount,
  (daily_rate * rental_duration_days * (1 - discount_percent / 100) + penalty_fee) as total_validation
from
  bronze.rental_cars_vehicle
where
  total_amount != (daily_rate * rental_duration_days * (1 - discount_percent / 100)+ penalty_fee)


## 2.2 Summary from Quality Check
base on quality chech above, there are few issues
1. sum of customer_name is not equal with sum of customer_phone
2. penalty_fee has not been included in total_amount

In [0]:
%sql
-- chek phone_number distribution
select
  customer_name,
  count(distinct customer_phone) as jumlah_phone,
  collect_list(distinct customer_phone) as daftar_phone
from
  bronze.rental_cars_vehicle
group by 1
having count(distinct customer_phone) > 1
order by jumlah_phone desc

## 2.3 Transform
there are few transformation that will do
1. just put 1 phone number for each custumer (lastes_phone number will used)
2. There is inconsitency information from customer data, one name have many phone number, email or addres in this transformation value that most appear will use
3. Add penalty_fee in total_amount

In [0]:
from pyspark.sql.functions import col, initcap, trim, regexp_replace, when, row_number, desc, last
from pyspark.sql.window import Window

# 1. read data from bronze layer
df=spark.table('bronze.rental_cars_vehicle')


# 2. take last phone_number
## a. use window for take last phone_number
w = Window.partitionBy('customer_name', 'customer_email') \
    .orderBy(desc('rental_date'), desc('transaction_id'))

## b. mark last phone_number
df = df.withColumn(
    'latest_phone_flag',
    row_number().over(w) == 1)

## c. create new field for new_customer_phone as phone_number
df = df.withColumn(
    'customer_phone_latest_temp',
    when(col('latest_phone_flag')== True, col('customer_phone')).otherwise(None))

## d. fill new_customer_phone with last phone_number
fill= Window.partitionBy('customer_name', 'customer_email') \
                    .orderBy("rental_date") \
                    .rowsBetween(Window.unboundedPreceding, Window.currentRow)
    

df = df.withColumn(
    'customer_phone_clean',
    last('customer_phone_latest_temp', ignorenulls=True).over(w))


# 3. add penalty_fee to total_amount
df = df.withColumn(
    'total_amount_calculated', 
    col('daily_rate') * col('rental_duration_days') * (1 - col('discount_percent') / 100) + col('penalty_fee'))

# 4. select field that neeede into silver layer
silver= df.select(
    "transaction_id",
    "rental_date",
    "customer_name",
    col("customer_phone_clean").alias("customer_phone"),
    "customer_email",
    "customer_city",
    "customer_age",
    "customer_gender",
    "vehicle_id",
    "vehicle_plate",
    "vehicle_brand",
    "vehicle_model",
    "vehicle_year",
    "vehicle_type",
    "vehicle_fuel",
    "pickup_location",
    "return_location",
    "pickup_location_city",
    "return_location_city",
    "rental_duration_days",
    "daily_rate",
    "discount_percent",
    "payment_method",
    "payment_status",
    "include_driver",
    "driver_name",
    "expected_return_date",
    "actual_return_date",
    "is_late_return",
    "penalty_fee",
    col('total_amount_calculated').alias("total_amount")
)

# 5. save to silver layer
silver.write.mode('overwrite') \
    .option('overwriteSchema', 'true') \
    .saveAsTable('silver.rental_cars_vehicle')

## 2.4 Quality Check of Silver Data

In [0]:
# cek issue data
## cek unique value

from pyspark.sql.functions import countDistinct

unique_values = silver.select([
    countDistinct(c).alias(c) for c in silver.columns
]).show(10, vertical=True)

unique_values

In [0]:
%sql
-- # check issue data
-- ## cek total_amount

select  
  is_late_return,
  total_amount,
  (daily_rate * rental_duration_days * (1 - discount_percent / 100) + penalty_fee) as total_validation
from
  silver.rental_cars_vehicle
where
  total_amount != (daily_rate * rental_duration_days * (1 - discount_percent / 100)+ penalty_fee)


# 3. Gold Layer

In this layer, data modeling was performed using star scheme. The data will be divided into fact and dimensions table
- dim_customer = customer information
- dim_vehicle = vehicle information
- dim_location = location information
- dim_date = date information
- rental_fact = rental transaction table


## 3.1 dim_customer

In [0]:
%sql
create table gold.dim_customer as 
select distinct
  row_number() over(order by customer_name) as customer_id,
  customer_name,
  customer_phone,
  customer_email,
  customer_city,
  customer_age,
  customer_gender
from
  silver.rental_cars_vehicle

In [0]:
%sql
select * from gold.dim_customer
limit (5)

In [0]:
%sql
-- Liat customer mana yang punya data beda
select 
    customer_name,
    count(distinct customer_city) as jumlah_kota,
    count(distinct customer_age) as jumlah_usia,
    count(distinct customer_gender) as jumlah_gender
from silver.rental_cars_vehicle
group by customer_name
having jumlah_kota > 1 or jumlah_usia > 1 or jumlah_gender > 1
order by jumlah_kota desc;

## 3.2 dim_vehicle

In [0]:
%sql
create table gold.dim_vehicle
select distinct
  vehicle_id,
  vehicle_plate,
  vehicle_brand,
  vehicle_model,
  vehicle_year,
  vehicle_type,
  vehicle_fuel
from
  silver.rental_cars_vehicle
    

In [0]:
%sql
select * from gold.dim_vehicle
limit (5)

## 3.3 dim_location

In [0]:
%sql
create or replace table gold.dim_location
select distinct
  row_number() over(order by location_name)as location_id,
  location_name,
  location_city
from
  (
    select 
      pickup_location as location_name, 
      pickup_location_city as location_city 
    from 
      silver.rental_cars_vehicle
    union
    select 
      return_location, 
      return_location_city 
    from 
      silver.rental_cars_vehicle
  )

  

In [0]:
%sql
    
select * from gold.dim_location
limit (5)

## 3.4 dim_date

In [0]:
%sql
create table gold.dim_date
with dates as(
  select 
    distinct rental_date as date 
  from 
    silver.rental_cars_vehicle
  union
  select distinct expected_return_date 
  from  
    silver.rental_cars_vehicle
  union
  select distinct actual_return_date 
  from  
    silver.rental_cars_vehicle
)
select
  date,
  YEAR(date) as year,
  MONTH(date) as month,
  DAY(date) as day,
  QUARTER(date) as quarter,
  weekofyear(date) as weekofyear,
  date_format(date,
  'EEEE') as dayofweekname,
  case when dayofweek(date) in (1, 7)then True else False end as is_weekend
from
  dates
where date is not null


In [0]:
%sql
select * from gold.dim_date
limit(5)

## 3.5 fact_rental

In [0]:
%sql
create table gold.fact_rental as
select 
    r.transaction_id,
    c.customer_id,
    v.vehicle_id,
    l_pickup.location_id as pickup_location_key,
    l_return.location_id as return_location_key,
    d_rental.date as rental_date,
    d_expected.date as expected_return_date,
    d_actual.date as actual_return_date,
    r.rental_duration_days,
    r.daily_rate,
    r.discount_percent,
    r.total_amount,
    r.payment_method,
    r.payment_status,
    r.include_driver,
    r.driver_name,
    r.is_late_return,
    r.penalty_fee
from 
  silver.rental_cars_vehicle r
left join gold.dim_customer c 
  on r.customer_name = c.customer_name
left join gold.dim_vehicle v 
  on r.vehicle_id = v.vehicle_id
left join gold.dim_location l_pickup 
  on r.pickup_location = l_pickup.location_name and r.pickup_location_city = l_pickup.location_city
left join gold.dim_location l_return 
  on r.return_location = l_return.location_name and r.return_location_city = l_return.location_city
left join gold.dim_date d_rental 
  on r.rental_date = d_rental.date
left join gold.dim_date d_expected 
  on r.expected_return_date = d_expected.date
left join gold.dim_date d_actual 
  on r.actual_return_date = d_actual.date;

In [0]:
%sql
select * from gold.fact_rental
limit (5)